# 02 - Graph Visualization and Recommendation Walkthrough

This notebook focuses on:
- visualizing top `NEXT` transitions from Neo4j
- optional recommendation examples
- simple reasoning traces for HSP and RIC
            


In [ ]:
from pathlib import Path
import sys
import math
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from database.driver import Neo4jDriver
            


## Fetch Top Transitions from Neo4j


In [ ]:
def fetch_top_next_transitions(limit=40):
    query = '''
    MATCH (src:Item)-[r:NEXT]->(dst:Item)
    RETURN src.item_id AS src, dst.item_id AS dst, r.weight AS w
    ORDER BY w DESC
    LIMIT $limit
    '''
    d = Neo4jDriver()
    try:
        rows = d.read_query(query, {'limit': int(limit)})
    finally:
        d.close()
    return rows

rows = fetch_top_next_transitions(limit=40)
print(f'loaded transitions: {len(rows)}')
rows[:5]
            


## Plot Transition Graph (Top Edges)


In [ ]:
def plot_transition_graph(edges):
    if not edges:
        print('No edges to plot.')
        return

    nodes = sorted({int(e['src']) for e in edges} | {int(e['dst']) for e in edges})
    weights = [float(e['w']) for e in edges]
    w_min, w_max = min(weights), max(weights)

    # Circular layout without extra dependencies.
    coords = {}
    n = len(nodes)
    for i, node in enumerate(nodes):
        angle = 2 * math.pi * i / max(n, 1)
        coords[node] = (math.cos(angle), math.sin(angle))

    plt.figure(figsize=(10, 10))

    for e in edges:
        s = int(e['src'])
        t = int(e['dst'])
        w = float(e['w'])
        x1, y1 = coords[s]
        x2, y2 = coords[t]

        alpha = 0.2 if w_max == w_min else 0.2 + 0.8 * ((w - w_min) / (w_max - w_min))
        lw = 0.5 if w_max == w_min else 0.5 + 3.5 * ((w - w_min) / (w_max - w_min))

        plt.plot([x1, x2], [y1, y2], alpha=alpha, linewidth=lw)

    xs = [coords[n][0] for n in nodes]
    ys = [coords[n][1] for n in nodes]
    plt.scatter(xs, ys, s=90)

    for node, (x, y) in coords.items():
        plt.text(x, y, str(node), fontsize=8, ha='center', va='center')

    plt.title('Top NEXT Transitions (Neo4j)')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

plot_transition_graph(rows)
            


## Example Recommendations + Reasoning


In [ ]:
from src.evaluation.tune_hyperparams import load_sessions
from src.inference.recommender import Recommender

sessions = load_sessions('data/processed/test_sessions.parquet', max_sessions=1)
example_full = sessions[0]
example_input = example_full[:-1][-5:]  # keep last few context items
example_target = example_full[-1]

print('example input:', example_input)
print('ground truth next item:', example_target)

rec = Recommender(top_k=10)
try:
    hsp_recs = rec.hsp_predict(example_input)
    ric_recs = rec.ric_predict(example_input)

    print('
HSP top-10:', hsp_recs)
    print('RIC top-10:', ric_recs)

    last_item = example_input[-1]
    hsp_reason = sorted(rec.next_edges.get(last_item, {}).items(), key=lambda x: x[1], reverse=True)[:10]
    ric_reason = sorted(rec.cooccurs.get(last_item, {}).items(), key=lambda x: x[1], reverse=True)[:10]

    print(f'
Top NEXT neighbors from last item {last_item} (HSP signal):')
    print(hsp_reason)

    print(f'
Top CO_OCCURS neighbors from last item {last_item} (RIC signal):')
    print(ric_reason)
finally:
    rec.close()
            
